### Cell 1: Environment Setup and Dependency Installation
This cell performs a clean installation of all necessary Python packages and system dependencies required for Chatterbox TTS to run correctly. It's crucial to run this cell first and allow the kernel to restart as instructed to ensure a pristine environment.

In [ ]:
# Emergency cleanup cell - run this if still having issues
!pip uninstall -y torch torchvision torchaudio transformers chatterbox-tts -y
!pip cache purge
print("Cleanup complete. Now run Cell 1.")

import subprocess
import sys
import os
from pathlib import Path

def run_command(command, description=""):
    """Run a command and handle errors gracefully"""
    print(f"Running: {description if description else command}")
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Warning: {description} failed with return code {result.returncode}")
            print(f"stderr: {result.stderr}")
            print(f"stdout: {result.stdout}")
            return False
        else:
            print(f"Success: {description}")
            return True
    except Exception as e:
        print(f"Error running command: {e}")
        return False

print(f"Python version: {sys.version}")

# Update pip first
run_command("pip install --upgrade pip", "Upgrading pip")

# CRITICAL: Complete uninstall of all related packages
print("🧹 Cleaning existing installations...")
run_command("pip uninstall -y torch torchvision torchaudio transformers chatterbox-tts accelerate huggingface-hub diffusers torchao perth resemble-perth", "Removing all related packages")

# CRITICAL FIX: Install PyTorch from PyPI (NOT from CPU index) to get full build with torch.fx
print("📦 Installing PyTorch 2.5.0 from PyPI...")
run_command(
    "pip install torch==2.5.0 torchaudio==2.5.0",
    "Installing PyTorch 2.5.0 (full build with torch.fx)"
)

# Install exact versions required by Chatterbox
print("📦 Installing Chatterbox dependencies...")
run_command("pip install transformers==4.46.3", "Installing transformers 4.46.3")
run_command("pip install diffusers==0.29.0", "Installing diffusers 0.29.0")
run_command("pip install huggingface_hub>=0.23.0", "Installing huggingface_hub")
run_command("pip install accelerate>=0.25.0", "Installing accelerate")

# Install git-lfs
run_command("apt update && apt install -y git-lfs", "Installing git-lfs")

# Install other required dependencies
print("📦 Installing audio processing libraries...")
run_command("pip install 'numpy>=1.24.0,<1.26.0' librosa==0.11.0 safetensors soundfile scipy", "Installing dependencies")

# CRITICAL FIX: Install resemble-perth (not just 'perth')
print("📦 Installing resemble-perth (watermarker)...")
run_command("pip install resemble-perth", "Installing resemble-perth watermarker")

# Install s3tokenizer and conformer
print("📦 Installing s3tokenizer and conformer...")
run_command("pip install s3tokenizer conformer", "Installing s3tokenizer and conformer")

# Install chatterbox-tts without dependencies
print("📦 Installing Chatterbox TTS...")
run_command("pip install chatterbox-tts --no-deps", "Installing Chatterbox TTS without dependencies")

# Fix protobuf
print("🔧 Fixing protobuf...")
run_command("pip uninstall -y protobuf", "Uninstalling protobuf")
run_command("pip install protobuf==3.20.3", "Installing protobuf 3.20.3")

print("\n✅ Installation complete!")
print("🔄 Restarting kernel...")
print("⚠️  WAIT for kernel restart, then run Cell 2!")
get_ipython().kernel.do_shutdown(True)

### Cell 2: Installation Verification and Model/Drive Setup
This cell verifies that all required libraries (PyTorch, Transformers, ChatterboxTTS) are correctly installed and can be imported. It also sets up Google Drive for saving generated audio and loads the Chatterbox TTS model. This cell should be run after Cell 1 has completed and the kernel has restarted.

In [ ]:
# Verify installation - import in specific order to avoid circular imports
import sys
print("🔍 Verifying installation...\n")

# Import torch first
try:
    import torch
    import torchaudio
    print(f"✅ PyTorch {torch.__version__} imported successfully")
    print(f"   CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA version: {torch.version.cuda}")
except ImportError as e:
    print(f"❌ PyTorch import error: {e}")
    print("   Please run Cell 1 again")
    sys.exit(1)

# Check if torchvision is installed (it shouldn't be)
try:
    import torchvision
    print(f"⚠️  Warning: torchvision {torchvision.__version__} is installed")
    print("   This may cause conflicts. Consider uninstalling it.")
except ImportError:
    print("✅ torchvision not installed (good - not needed for Chatterbox)")

# Import transformers
try:
    import transformers
    print(f"✅ Transformers {transformers.__version__} imported successfully")
except ImportError as e:
    print(f"❌ Transformers import error: {e}")
    sys.exit(1)

# Import Chatterbox TTS - this is the critical test
try:
    from chatterbox.tts import ChatterboxTTS
    print("✅ ChatterboxTTS imported successfully")
    print("\n🎉 All imports successful! You can proceed to Cell 3.")
except Exception as e:
    print(f"❌ ChatterboxTTS import error: {e}")
    print("\n🔧 Troubleshooting:")
    print("   1. Make sure kernel was restarted after Cell 1")
    print("   2. If error persists, run this command in a new cell:")
    print("      !pip uninstall -y torchvision")
    print("   3. Then restart kernel and try Cell 2 again")
    sys.exit(1)

    from google.colab import drive
import os

def setup_drive():
    try:
        drive.mount('/content/drive')
        drive_path = '/content/drive/MyDrive/Chatterbox'
        os.makedirs(drive_path, exist_ok=True)
        print(f"✅ Drive setup complete: {drive_path}")
        return drive_path
    except Exception as e:
        print(f"❌ Drive setup failed: {e}")
        return None

DRIVE_PATH = setup_drive()


import torch
from chatterbox.tts import ChatterboxTTS

def load_model():
    try:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Loading model on device: {device}")

        model = ChatterboxTTS.from_pretrained(device=device)
        print("✅ Model loaded successfully")
        return model

    except Exception as e:
        print(f"❌ Model loading failed: {e}")
        print("Trying CPU fallback...")

        try:
            model = ChatterboxTTS.from_pretrained(device="cpu")
            print("✅ Model loaded successfully on CPU")
            return model
        except Exception as e2:
            print(f"❌ CPU fallback also failed: {e2}")
            raise e2

model = load_model()


class ChatterboxConfig:
    def __init__(self):
        self.exaggeration = 0.5
        self.cfg_weight = 0.5
        self.max_chunk_words = 50
        self.voice_sample_path = None

    def get_preset(self, preset_name):
        presets = {
            "neutral": {"exaggeration": 0.5, "cfg_weight": 0.5},
            "calm": {"exaggeration": 0.3, "cfg_weight": 0.6},
            "expressive": {"exaggeration": 0.7, "cfg_weight": 0.4},
            "dramatic": {"exaggeration": 1.0, "cfg_weight": 0.3},
            "storytelling": {"exaggeration": 0.8, "cfg_weight": 0.4},
            "audiobook": {"exaggeration": 0.4, "cfg_weight": 0.6},
            "fast_speaker": {"exaggeration": 0.6, "cfg_weight": 0.3},
        }
        return presets.get(preset_name, presets["storytelling"])

config = ChatterboxConfig()

config.exaggeration = 0.75
config.cfg_weight = 0.3
config.max_chunk_words = 40

print(f"🎛️ Current settings:")
print(f"   Exaggeration: {config.exaggeration}")
print(f"   CFG Weight: {config.cfg_weight}")
print(f"   Chunk size: {config.max_chunk_words} words")

def setup_voice_cloning():
    print("🎤 VOICE CLONING SETUP")
    print("=" * 50)
    print("For best results, your voice sample should:")
    print("• Be at least 10 seconds long (ideally 15-30 seconds)")
    print("• Be in WAV format")
    print("• Have clear, consistent audio quality")
    print("• Contain natural speech (avoid reading lists/monotone)")
    print("• Be recorded in a quiet environment")
    print("• Match the speaking style you want to generate")
    print()

    if DRIVE_PATH:
        sample_path = f"{DRIVE_PATH}/voice_sample.wav"
        print(f"📁 Upload your voice sample to: {sample_path}")
        print("   Or use Google Colab's file upload feature")

        if os.path.exists(sample_path):
            print(f"✅ Voice sample found: {sample_path}")
            config.voice_sample_path = sample_path
            return sample_path
        else:
            print(f"⚠️  No voice sample found at {sample_path}")
            print("   Voice cloning will be disabled")
            return None
    else:
        print("❌ Google Drive not mounted - voice cloning disabled")
        return None

voice_sample = setup_voice_cloning()

def split_into_chunks(text, max_words=100):
    sentences = text.strip().replace('\n', ' ').split('.')
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current_chunk = ""
    current_word_count = 0

    for sentence in sentences:
        sentence_words = sentence.split()

        if current_word_count + len(sentence_words) > max_words and current_chunk:
            chunks.append(current_chunk.strip() + ".")
            current_chunk = sentence
            current_word_count = len(sentence_words)
        else:
            if current_chunk:
                current_chunk += ". " + sentence
            else:
                current_chunk = sentence
            current_word_count += len(sentence_words)

    if current_chunk:
        chunks.append(current_chunk.strip() + ".")

    return chunks

def estimate_processing_time(text, words_per_minute=150):
    word_count = len(text.split())
    estimated_minutes = word_count / words_per_minute
    return word_count, estimated_minutes

sample_text = """
This is a test of the enhanced Chatterbox TTS system.
The system now includes advanced controls for voice quality and expression.
You can adjust parameters like exaggeration and CFG weight for different effects.
"""

chunks = split_into_chunks(sample_text, config.max_chunk_words)
word_count, time_est = estimate_processing_time(sample_text)

print(f"📊 Text Analysis:")
print(f"   Total words: {word_count}")
print(f"   Estimated time: {time_est:.1f} minutes")
print(f"   Number of chunks: {len(chunks)}")
print(f"   Chunk preview: '{chunks[0][:50]}...'")


def generate_speech(text, config, model, output_filename="generated_speech.wav"):
    print("🎙️ STARTING SPEECH GENERATION")
    print("=" * 50)

    chunks = split_into_chunks(text, config.max_chunk_words)
    word_count, time_est = estimate_processing_time(text)

    print(f"📝 Text: {word_count} words, {len(chunks)} chunks")
    print(f"⏱️  Estimated time: {time_est:.1f} minutes")
    print(f"🎛️ Settings: exaggeration={config.exaggeration}, cfg_weight={config.cfg_weight}")

    if config.voice_sample_path and os.path.exists(config.voice_sample_path):
        print(f"🎤 Using voice cloning: {config.voice_sample_path}")
    else:
        print("🤖 Using default voice (no cloning)")

    print("\n🔄 Processing chunks...")

    wav_tensors = []

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}: '{chunk[:50]}...'")

        try:
            gen_params = {
                "text": chunk,
                "exaggeration": config.exaggeration,
                "cfg_weight": config.cfg_weight
            }

            if config.voice_sample_path and os.path.exists(config.voice_sample_path):
                gen_params["audio_prompt_path"] = config.voice_sample_path

            wav = model.generate(**gen_params)
            wav_tensors.append(wav)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"❌ Error in chunk {i+1}: {e}")
            continue

    if wav_tensors:
        print(f"\n💾 Saving {len(wav_tensors)} audio chunks...")
        full_audio = torch.cat(wav_tensors, dim=1)

        if DRIVE_PATH:
            output_path = f"{DRIVE_PATH}/{output_filename}"
            torchaudio.save(output_path, full_audio, model.sr)
            print(f"✅ Audio saved to: {output_path}")

            local_path = f"/content/{output_filename}"
            torchaudio.save(local_path, full_audio, model.sr)
            print(f"📱 Local copy: {local_path}")

            return output_path, local_path
        else:
            local_path = f"/content/{output_filename}"
            torchaudio.save(local_path, full_audio, model.sr)
            print(f"✅ Audio saved to: {local_path}")
            return local_path, local_path
    else:
        print("❌ No audio was generated")
        return None, None

your_text = """

Welcome to ChatterBox

"""

output_path, local_path = generate_speech(your_text, config, model)


import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np

def play_and_analyze_audio(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        print("❌ No audio file to play")
        return

    print(f"🔊 Playing audio: {audio_path}")

    ipd.display(ipd.Audio(audio_path))

    try:
        waveform, sample_rate = torchaudio.load(audio_path)
        duration = waveform.shape[1] / sample_rate

        print(f"📊 Audio Analysis:")
        print(f"   Duration: {duration:.2f} seconds")
        print(f"   Sample Rate: {sample_rate} Hz")
        print(f"   Channels: {waveform.shape[0]}")

        plt.figure(figsize=(12, 4))
        plt.plot(waveform[0].numpy())
        plt.title("Generated Audio Waveform")
        plt.xlabel("Sample")
        plt.ylabel("Amplitude")
        plt.grid(True, alpha=0.3)
        plt.show()

    except Exception as e:
        print(f"❌ Error analyzing audio: {e}")

if local_path:
    play_and_analyze_audio(local_path)